# 17 · Gráficas con pandas

pandas incluye `.plot()`, que usa Matplotlib por debajo. Con una línea se obtiene una gráfica.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

## Datos de ejemplo

In [ ]:
mensual = pd.DataFrame({
    "mes": ["Ene", "Feb", "Mar", "Abr", "May", "Jun"],
    "ventas": [120, 135, 150, 145, 170, 190],
    "gastos": [100, 110, 120, 125, 130, 140],
})
mensual

## Gráfica de líneas

Ideal para ver **evolución en el tiempo**.

In [ ]:
mensual.plot(x="mes", y="ventas", kind="line", marker="o", title="Ventas por mes")
plt.ylabel("Miles de pesos")
plt.show()

In [ ]:
# Varias columnas a la vez
mensual.plot(x="mes", y=["ventas", "gastos"], marker="o", title="Ventas vs. gastos")
plt.show()

## Gráfica de barras

Ideal para **comparar categorías**.

In [ ]:
mensual.plot(x="mes", y=["ventas", "gastos"], kind="bar", title="Ventas y gastos", rot=0)
plt.show()

In [ ]:
# Barras horizontales
mensual.plot(x="mes", y="ventas", kind="barh", color="seagreen", legend=False)
plt.show()

## Datos agrupados + gráfica

Usamos las ventas generadas en el cuaderno 16 (`ventas.csv`). Si no lo ejecutaste, se regeneran aquí.

In [ ]:
import os, random

if not os.path.exists("ventas.csv"):
    random.seed(1)
    v = pd.DataFrame({
        "mes": random.choices(["Ene", "Feb", "Mar"], k=90),
        "tienda": random.choices(["Norte", "Centro", "Sur"], k=90),
        "producto": random.choices(["Lápiz", "Cuaderno", "Mochila"], k=90),
        "unidades": [random.randint(1, 20) for _ in range(90)],
    })
    v["ingreso"] = v["unidades"] * v["producto"].map({"Lápiz": 5, "Cuaderno": 25, "Mochila": 150})
    v.to_csv("ventas.csv", index=False)

ventas = pd.read_csv("ventas.csv")
ventas.head()

In [ ]:
por_tienda = ventas.groupby("tienda")["ingreso"].sum()
por_tienda.plot(kind="bar", title="Ingreso por tienda", rot=0)
plt.ylabel("Ingreso")
plt.show()

In [ ]:
# Barras apiladas: tienda x producto
tabla = ventas.pivot_table(index="tienda", columns="producto", values="ingreso", aggfunc="sum")
tabla.plot(kind="bar", stacked=True, title="Ingreso por tienda y producto", rot=0)
plt.show()

## Gráfica circular (pastel)

Para mostrar **proporciones** de un total (con pocas categorías).

In [ ]:
ventas.groupby("producto")["ingreso"].sum().plot(kind="pie", autopct="%1.1f%%", ylabel="")
plt.title("Participación por producto")
plt.show()

## Histograma

Muestra **cómo se distribuyen** los valores de una columna numérica.

In [ ]:
ventas["unidades"].plot(kind="hist", bins=10, edgecolor="white", title="Distribución de unidades")
plt.xlabel("Unidades por venta")
plt.show()

## Diagrama de dispersión (scatter)

Relación entre **dos** variables numéricas.

In [ ]:
ventas.plot(kind="scatter", x="unidades", y="ingreso", alpha=0.6, title="Unidades vs. ingreso")
plt.show()

## Caja (boxplot)

Resume mediana, cuartiles y valores atípicos.

In [ ]:
ventas.boxplot(column="unidades", by="tienda")
plt.suptitle("")
plt.title("Unidades por tienda")
plt.show()

## Varias gráficas en una figura

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
mensual.plot(x="mes", y="ventas", ax=ejes[0], marker="o", title="Líneas")
mensual.plot(x="mes", y="ventas", ax=ejes[1], kind="bar", title="Barras", rot=0)
plt.tight_layout()
plt.show()

## Guardar una gráfica

In [ ]:
ax = por_tienda.plot(kind="bar", rot=0)
ax.figure.savefig("ingreso_por_tienda.png", dpi=150, bbox_inches="tight")
plt.show()

## ¿Qué gráfica elegir?

| Quiero mostrar... | Tipo |
|-------------------|------|
| Evolución en el tiempo | `line` |
| Comparar categorías | `bar` / `barh` |
| Proporciones | `pie` (pocas categorías) |
| Distribución de una variable | `hist`, `box` |
| Relación entre dos variables | `scatter` |

## 🎛️ Constructor de gráficas

Elige el tipo de gráfica, la tienda y la métrica.

In [ ]:
import ipywidgets as widgets

tipo = widgets.ToggleButtons(options=["bar", "barh", "line", "pie"], description="Tipo:")
columna = widgets.Dropdown(options=["ingreso", "unidades"], description="Métrica:")
filtro_tienda = widgets.Dropdown(options=["Todas"] + sorted(ventas["tienda"].unique()), description="Tienda:")
agrupar = widgets.Dropdown(options=["producto", "mes", "tienda"], description="Agrupar por:")
out = widgets.Output()

def graficar(_=None):
    datos = ventas if filtro_tienda.value == "Todas" else ventas[ventas["tienda"] == filtro_tienda.value]
    serie = datos.groupby(agrupar.value)[columna.value].sum()
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6, 3.5))
        if tipo.value == "pie":
            serie.plot(kind="pie", autopct="%1.0f%%", ax=ax, ylabel="")
        else:
            serie.plot(kind=tipo.value, ax=ax)
        ax.set_title(f"{columna.value} por {agrupar.value} ({filtro_tienda.value})")
        plt.show()

for w in (tipo, columna, filtro_tienda, agrupar):
    w.observe(graficar, names="value")
display(tipo, widgets.HBox([columna, filtro_tienda, agrupar]), out)
graficar()

## Resumen

- `df.plot(kind=..., x=..., y=...)` crea la gráfica; `plt.show()` la muestra.
- Se combina con `groupby` para graficar resúmenes.
- Títulos y ejes: `title=`, `plt.xlabel()`, `plt.ylabel()`.
- Guarda con `figura.savefig("archivo.png")`.

🎉 **¡Terminaste el curso introductorio!** Ahora prueba con tus propios datos.